# 01 — Prepare Feature Datasets for Classification Experiment

This notebook builds all feature CSVs needed by `02_classify.ipynb`.

**Data pipeline** (fully self-contained):
1. Load VOSA training labels (binary classification targets)
2. Fetch XP coefficients from ESA Gaia DR3 archive via `clustertools`
3. Parse and L2-normalize XP coefficients → baseline features
4. Calibrate XP spectra via `gaiaxpy` → sampled wavelength–flux spectra
5. Fit orthogonal polynomials to sampled spectra → polynomial features
6. Save all feature sets and stratified train/test splits

**`clustertools` functions used:**
- `clustertools.data.gaia.query_xp_coefficients` — fetch raw XP data from ESA Gaia TAP
- `clustertools.spectra.xp.parse_xp_coefficients` — parse BP/RP into 110 float columns
- `clustertools.spectra.xp.l2_normalize` — row-wise L2 normalization
- `clustertools.spectra.polynomial.fit_polynomial` — orthogonal polynomial fitting
- `clustertools.spectra.polynomial.POLY_CLASSES` — available polynomial bases

**Outputs** (to `data/`):
- `og_xp.csv` — original 110 Gaia XP coefficients (L2-normalized baseline)
- `{basis}_{n}_L2.csv` — L2-normalized polynomial coefficients
- `{basis}_{n}_raw.csv` — raw (unnormalized) polynomial coefficients
- `splits.json` — 10 stratified train/test splits (80/20)

**Polynomial bases**: Chebyshev, Hermite (probabilists'), Laguerre, Legendre  
**Dimensionalities**: 10, 20, 30, 40, 50  

**Normalization as experimental variable**: polynomial features are saved in both
raw and L2-normalized variants to test the impact of L2 normalization on
classification performance (L2 may destroy amplitude information that
polynomial coefficients encode).

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from clustertools.data.gaia import query_xp_coefficients
from clustertools.spectra.xp import parse_xp_coefficients, l2_normalize
from clustertools.spectra.polynomial import fit_polynomial, POLY_CLASSES
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

Some IP addresses of users launching heavy query showers have temporarily been disabled. Please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk) for advice. Workaround solutions for the Gaia Archive issues following the infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


In [2]:
# ── Configuration ──
EXPERIMENT_DIR = Path.cwd() if Path("data").exists() else Path("transformation_experiment")
POC_DIR = Path("transformation_poc") if Path("transformation_poc").exists() else Path("..") / "transformation_poc"
DATA_OUT = EXPERIMENT_DIR / "data"
CACHE_DIR = EXPERIMENT_DIR / "cache"
DATA_OUT.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

LABELS_PATH = POC_DIR / "VOSA_labels_training.csv"

N_COEFFS_LIST = [10, 20, 30, 40, 50]
N_SPLITS = 10
TEST_SIZE = 0.20

print("Experiment dir:", EXPERIMENT_DIR.resolve())
print("Labels path:   ", LABELS_PATH.resolve())
print("Output dir:    ", DATA_OUT.resolve())
print("Cache dir:     ", CACHE_DIR.resolve())

Experiment dir: /Users/erikak/Documents/uni/bakalauras/kodas/transformation_experiment
Labels path:    /Users/erikak/Documents/uni/bakalauras/kodas/transformation_poc/VOSA_labels_training.csv
Output dir:     /Users/erikak/Documents/uni/bakalauras/kodas/transformation_experiment/data
Cache dir:      /Users/erikak/Documents/uni/bakalauras/kodas/transformation_experiment/cache


## 1. Load training labels

Binary classification labels from the [Hot subdwarfs / GaiaXP](https://github.com/mambr-astro/Hot_sds_GaiaXP) dataset.
Each star has a Gaia DR3 `source_id` and a VOSA-derived binary label.

In [3]:
df_labels = pd.read_csv(LABELS_PATH)
df_labels = df_labels.rename(columns={"GaiaDR3": "source_id", "VOSA": "y"})
source_ids = df_labels["source_id"].tolist()
labels = df_labels["y"].values

print(f"Labels: {len(df_labels)} stars")
print(f"Class balance: {dict(df_labels['y'].value_counts().sort_index())}")

Labels: 2815 stars
Class balance: {0: np.int64(2257), 1: np.int64(558)}


## 2. Fetch and process OG XP coefficients (baseline)

Query `gaiadr3.xp_continuous_mean_spectrum` via ESA Gaia TAP using
`clustertools.data.gaia.query_xp_coefficients`, then parse the raw
BP/RP coefficient arrays into 110 float columns and L2-normalize.

In [4]:
xp_cache = CACHE_DIR / "xp_parsed_110d.csv"

if xp_cache.exists():
    print(f"Loading cached parsed XP coefficients from {xp_cache.name}")
    df_xp_parsed = pd.read_csv(xp_cache)
else:
    print(f"Fetching XP coefficients for {len(source_ids)} sources from ESA Gaia TAP...")
    df_xp_raw = query_xp_coefficients(source_ids, chunk_size=400)
    print(f"Received {len(df_xp_raw)} rows from archive")

    df_xp_parsed = parse_xp_coefficients(df_xp_raw)
    df_xp_parsed.to_csv(xp_cache, index=False)
    print(f"Parsed and cached → {xp_cache.name}")

print(f"Parsed XP coefficients: {df_xp_parsed.shape[0]} stars × {df_xp_parsed.shape[1] - 1} features")

# L2-normalize
df_xp_l2 = l2_normalize(df_xp_parsed)

# Merge with labels and save baseline
df_og = df_xp_l2.merge(df_labels, on="source_id", how="inner")
print(f"OG XP with labels: {df_og.shape[0]} stars, {df_og.shape[1] - 2} features")
print(f"Class balance: {dict(df_og['y'].value_counts().sort_index())}")

og_out = DATA_OUT / "og_xp.csv"
df_og.to_csv(og_out, index=False)
print(f"\nSaved baseline: {og_out.name}")

21:59:13 [clustertools.data.gaia] INFO: query_xp_coefficients: chunk 1/8 (400 source_ids)


Fetching XP coefficients for 2815 sources from ESA Gaia TAP...


21:59:16 [clustertools.data.gaia] WARNING: TAP query failed (attempt 1/5), retrying in 10s: Error 500:
null


500 Error 500:
null


21:59:29 [clustertools.data.gaia] WARNING: TAP query failed (attempt 2/5), retrying in 20s: Error 500:
null


...

22:00:35 [clustertools.data.gaia] WARNING: TAP query failed (attempt 4/5), retrying in 80s: Error 500:
null


500 Error 500:
null


22:01:58 [clustertools.data.gaia] ERROR: query_xp_coefficients failed at chunk 1/8
Traceback (most recent call last):
  File "/Users/erikak/Documents/uni/bakalauras/clustertools/clustertools/data/gaia.py", line 303, in query_xp_coefficients
    df_chunk = _run_tap_with_retry(adql, max_retries=max_retries)
  File "/Users/erikak/Documents/uni/bakalauras/clustertools/clustertools/data/gaia.py", line 54, in _run_tap_with_retry
    job = Gaia.launch_job_async(adql)
  File "/Users/erikak/Documents/uni/bakalauras/kodas/.venv/lib/python3.14/site-packages/astroquery/gaia/core.py", line 1154, in launch_job_async
    return TapPlus.launch_job_async(self, query=query,
           ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
                                    name=name,
                                    ^^^^^^^^^^
    ...<7 lines>...
                                    autorun=autorun,
                                    ^^^^^^^^^^^^^^^^
                                    format_with_results_comp

500 Error 500:
null


QueryError: Gaia XP coefficients query failed: Error 500:
null

## 3. Fetch sampled spectra for polynomial fitting

Use `gaiaxpy.calibrate` to convert internal XP coefficients into
externally calibrated sampled spectra (flux vs wavelength, 336–1020 nm).
These serve as the target signal for polynomial approximation.

In [ ]:
spectra_cache = CACHE_DIR / "sampled_spectra.csv"

if spectra_cache.exists():
    print(f"Loading cached sampled spectra from {spectra_cache.name}")
    df_spectra = pd.read_csv(spectra_cache)
else:
    from gaiaxpy import calibrate as gaiaxpy_calibrate

    # Use source_ids that have XP coefficients
    xp_source_ids = df_xp_parsed["source_id"].tolist()
    print(f"Calibrating XP spectra for {len(xp_source_ids)} sources via gaiaxpy...")
    spectra_result, sampling = gaiaxpy_calibrate(xp_source_ids)

    wl_cols = [f"wl_{wl:.0f}" for wl in sampling]
    flux_matrix = np.vstack(spectra_result["flux"].values)

    df_spectra = pd.DataFrame(flux_matrix, columns=wl_cols)
    df_spectra.insert(0, "source_id", spectra_result["source_id"].values)
    df_spectra = df_spectra.merge(df_labels, on="source_id", how="inner")
    df_spectra.to_csv(spectra_cache, index=False)
    print(f"Cached → {spectra_cache.name}")

wl_cols = [c for c in df_spectra.columns if c.startswith("wl_")]
wavelengths = np.array([float(c.split("_")[1]) for c in wl_cols])
spectra_source_ids = df_spectra["source_id"].values
spectra_labels = df_spectra["y"].values
spectra_matrix = df_spectra[wl_cols].to_numpy(dtype=np.float64)

print(f"Sampled spectra: {spectra_matrix.shape[0]} stars, {spectra_matrix.shape[1]} wavelength bins")
print(f"Wavelength range: {wavelengths[0]:.0f} – {wavelengths[-1]:.0f} nm")

## 4. Generate polynomial features

For each (basis, n_coeffs), fit an orthogonal polynomial to each star's
spectrum using `clustertools.spectra.polynomial.fit_polynomial` and extract
the expansion coefficients as features.

Each combination is saved in **two variants**:
- `{basis}_{n}_raw.csv` — raw coefficients (preserves amplitude information)
- `{basis}_{n}_L2.csv` — L2-normalized coefficients (same preprocessing as OG XP)

In [ ]:
# clustertools uses capitalized names; map to lowercase for file output
BASIS_MAP = {
    "Chebyshev": "chebyshev",
    "Hermite":   "hermite",
    "Laguerre":  "laguerre",
    "Legendre":  "legendre",
}


def fit_all_stars(wavelengths, spectra_matrix, basis_name, n_coeffs):
    """Fit polynomial to every star using clustertools.spectra.polynomial.fit_polynomial."""
    n_stars = spectra_matrix.shape[0]
    coeff_matrix = np.zeros((n_stars, n_coeffs), dtype=np.float64)
    r2_values = np.zeros(n_stars, dtype=np.float64)

    for i in range(n_stars):
        result = fit_polynomial(
            wavelengths, spectra_matrix[i],
            basis=basis_name, n_coeffs=n_coeffs,
        )
        coeff_matrix[i] = result["coefficients"][:n_coeffs]
        r2_values[i] = result["metrics"]["R2"]

    return coeff_matrix, r2_values


def build_feature_df(source_ids, labels, coeff_matrix, n_coeffs):
    """Assemble a feature DataFrame with source_id, y, and coefficient columns."""
    col_names = [f"c{i:03d}" for i in range(n_coeffs)]
    df = pd.DataFrame(coeff_matrix, columns=col_names)
    df.insert(0, "y", labels)
    df.insert(0, "source_id", source_ids)
    return df

In [ ]:
reconstruction_quality = []

for basis_name, file_prefix in BASIS_MAP.items():
    for n_coeffs in N_COEFFS_LIST:
        print(f"  {file_prefix:10s} n={n_coeffs:3d} ... ", end="", flush=True)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            coeff_raw, r2_values = fit_all_stars(
                wavelengths, spectra_matrix, basis_name, n_coeffs,
            )

        median_r2 = np.median(r2_values)
        reconstruction_quality.append({
            "basis": file_prefix, "n_coeffs": n_coeffs,
            "median_r2": median_r2, "min_r2": np.min(r2_values),
        })

        # Build raw feature DataFrame
        df_feat_raw = build_feature_df(spectra_source_ids, spectra_labels, coeff_raw, n_coeffs)

        # L2-normalize using clustertools
        coeff_cols = [f"c{i:03d}" for i in range(n_coeffs)]
        df_feat_l2 = l2_normalize(df_feat_raw, coeff_cols=coeff_cols)

        # Save both variants
        raw_path = DATA_OUT / f"{file_prefix}_{n_coeffs}_raw.csv"
        l2_path  = DATA_OUT / f"{file_prefix}_{n_coeffs}_L2.csv"
        df_feat_raw.to_csv(raw_path, index=False)
        df_feat_l2.to_csv(l2_path, index=False)

        print(f"R²={median_r2:.6f}  → {raw_path.name}, {l2_path.name}")

print("\nDone. All polynomial features saved (raw + L2 variants).")

## 5. Reconstruction quality summary

Median R² across all stars for each (basis, n_coeffs) — shows how well
each polynomial representation captures the spectral shape.

In [ ]:
df_quality = pd.DataFrame(reconstruction_quality)

fig, ax = plt.subplots(figsize=(8, 4))
for basis in BASIS_MAP.values():
    subset = df_quality[df_quality["basis"] == basis]
    ax.plot(subset["n_coeffs"], subset["median_r2"], "o-", label=basis.capitalize())
ax.set_xlabel("Number of coefficients")
ax.set_ylabel("Median R²")
ax.set_title("Reconstruction quality by polynomial basis")
ax.legend()
ax.set_ylim(0.98, 1.001)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

df_quality.pivot(index="n_coeffs", columns="basis", values="median_r2").round(6)

## 6. Generate reproducible train/test splits

10 stratified 80/20 splits. With 10 paired observations the Wilcoxon
signed-rank test achieves a minimum p-value of ~0.002 (vs 0.0625 with 5 splits).

In [ ]:
og_labels = df_og["y"].values

splits = {}
sss = StratifiedShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)

for i, (train_idx, test_idx) in enumerate(sss.split(np.zeros(len(og_labels)), og_labels)):
    splits[f"seed_{i}"] = {
        "train": train_idx.tolist(),
        "test":  test_idx.tolist(),
    }
    n_pos_tr = og_labels[train_idx].sum()
    n_pos_te = og_labels[test_idx].sum()
    print(f"Split {i}: train={len(train_idx)} (pos={n_pos_tr}), test={len(test_idx)} (pos={n_pos_te})")

splits_path = DATA_OUT / "splits.json"
with open(splits_path, "w") as f:
    json.dump(splits, f, indent=2)

print(f"\nSaved {N_SPLITS} splits → {splits_path.name}")

In [ ]:
print("\nFiles in data/:")
for p in sorted(DATA_OUT.glob("*")):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:30s}  {size_kb:8.1f} KB")